In [ ]:
# ========== 练习说明（教学旁注）==========
# 本笔记本乐趣在于：复制后改超参数（epochs、batch_size），
# 在约 20,000 条测试数据设定下观察微调效果差异
# 复制调整模型更改超参数的乐趣，
# 20.000 个测试数据的 Epochs、批量大小


## 第 6 周练习：微调 GPT 做商品估价

### 练习目标

走通 OpenAI **Fine-tuning** 流程：准备 JSONL → 上传文件 → 创建微调作业 → 用微调模型推理并 `evaluate`。

### 和本课关系

| 概念 | 本笔记本位置 |
|------|----------------|
| Hub 数据集 `Item.from_hub` | 加载 train/val/test |
| messages（user/assistant） | `messages_for` / `test_messages_for` |
| Fine-tuning API | `openai.fine_tuning.jobs.*` |
| 统一评测 | `pricer.evaluator.evaluate` |

### 怎么跑

1. 确认能 `import pricer`（通常在 `week6` 目录下运行）
2. `.env` 配置 `HF_TOKEN` 与 `OPENAI_API_KEY`
3. 自上而下运行；创建微调作业会产生费用


In [ ]:
# ========== 导入：环境、Hub 登录、OpenAI、课程定价工具 ==========

# 导入 os：读环境变量
import os
# 导入 re：正则（本练习后续可扩展解析价格）
import re
# 导入 json：序列化 messages / 拼 JSONL
import json
# load_dotenv：从 .env 加载密钥，避免写死在代码里
from dotenv import load_dotenv
# login：HuggingFace Hub 登录
from huggingface_hub import login
# OpenAI 官方客户端
from openai import OpenAI
# Item：课程商品对象 + from_hub
from pricer.items  import Item
# evaluate：统一评估定价器
from pricer.evaluator import evaluate


In [ ]:
# ========== 环境：LITE_MODE + 加载密钥 + HF 登录 ==========

# True：用轻量 items_lite，省流量、省时间
LITE_MODE = True

# override=True：.env 覆盖进程里已有同名变量
load_dotenv(override=True)
# 必须有 HF_TOKEN，否则 KeyError（原逻辑如此）
hf_token = os.environ['HF_TOKEN']
# 登录 Hub；写入 git credential 方便后续 git 操作
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 从 Hub 拉取 train / val / test ==========

# 数据集所属用户（字符串保持原样）
username = "ed-donner"
# 按 LITE_MODE 选择 lite 或 full
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

# 一次加载三个划分
train, val, test = Item.from_hub(dataset)

# 打印规模，确认数据到手
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


## 准备微调子集与 OpenAI 客户端

下面会截取较小的 train/val 子集写 JSONL，再调用 Fine-tuning API（付费）。


In [ ]:
# ========== 创建 OpenAI 客户端（读环境变量里的 API Key） ==========
openai = OpenAI()


In [ ]:
# ========== 截取微调子集：250 条训练 + 50 条验证 ==========
# 子集小：省钱、作业更快；想提分可增大（费用也升）

fine_tune_train = train[:250]
fine_tune_validation = val[:50]


In [ ]:
# ========== 构造微调 messages：user 给估价指令，assistant 给真价 ==========
# 注意：message 多行字符串是 prompt，必须保持英文原文，不要翻译

def messages_for(item):
    message = f"""
    You are a product price estimation AI.
    Your task is to estimate the approximate market price in USD for a product based only on the provided description.
    Guidelines:
    - Use general global market knowledge to estimate the price.
    - Consider brand, materials, specifications, size, and category when mentioned.
    - If the description is vague, make a reasonable assumption based on common products in that category.
    - Do not explain your reasoning.
    - Do not include currency symbols or text.
    - Return only a single number representing the estimated price in USD.
    - Round to the nearest whole number.

    Output format:
    <number>

    Examples:

    Input: "Apple iPhone 14 Pro Max 256GB smartphone"
    Output:
    1100

    Input: "Wooden dining table for 6 people"
    Output:
    450

    Input: "Basic cotton t-shirt"
    Output:
    15
    """
    return [
        {"role": "user", "content": message},
        # assistant 侧放带两位小数的真价，作为监督标签
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]


In [ ]:
# ========== 把多条 Item 收成 JSONL 文本（每行一条 {"messages": ...}） ==========

def make_jsonl(items):
    result = ""
    for item in items:
        # 取出该样本的 messages 列表
        messages = messages_for(item)
        # 序列化为 JSON 数组字符串
        messages_str = json.dumps(messages)
        # 手工拼成 OpenAI 微调要求的行格式（注意结尾换行）
        result += '{"messages": ' + messages_str +'}\n'
    # 去掉末尾多余空白/换行
    return result.strip()


In [ ]:
# ========== 写盘：Items -> JSONL 文件 ==========

def write_jsonl(items, filename):
    # 文本模式写入目标路径
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)


In [ ]:
# ========== 写出训练集 JSONL ==========
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")


In [ ]:
# ========== 写出验证集 JSONL ==========
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")


In [ ]:
# ========== 上传训练文件到 OpenAI（purpose=fine-tune） ==========
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")


In [ ]:
# ========== 上传验证文件到 OpenAI ==========
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")


## 创建并监控微调作业

`model` / `seed` / `hyperparameters` / `suffix` 保持与原代码一致；改超参前先想清楚样本只有 250 条时的 batch 上限。


In [ ]:
# ========== 提交 fine-tuning job ==========
openai.fine_tuning.jobs.create(
    # 训练文件 id（上格 upload 返回）
    training_file=train_file.id,
    # 验证文件 id
    validation_file=validation_file.id,
    # 基座模型 id：字符串必须原样保留
    model="gpt-4.1-nano-2025-04-14",
    # 固定种子，便于复现
    seed=42,
    # 1 个 epoch、batch_size=1：样本只有 250 时很难把 batch 再加大
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    # hyperparameters={"n_epochs": 1, "batch_size": 1}，想法是增加batch size，但样本数只有250，所以我们不能将batch size增加超过1和epochs。
    # 后缀会出现在微调后模型名里，方便辨认
    suffix="pricer"
)


In [ ]:
# ========== 列出最近 1 条微调作业（看状态） ==========
openai.fine_tuning.jobs.list(limit=1)


In [ ]:
# ========== 取最近作业的 job_id ==========
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id


In [ ]:
# ========== 按 job_id 查询作业详情（状态、超参、结果模型名等） ==========
openai.fine_tuning.jobs.retrieve(job_id)


In [ ]:
# ========== 拉取该作业最近 10 条事件日志 ==========
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data


In [ ]:
# ========== 作业完成后读取 fine_tuned_model 名称 ==========
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model


In [ ]:
# ========== 推理用 prompt：只要 user，不带 assistant 真价 ==========
# prompt 英文原文必须保留

def test_messages_for(item):
    message = (f"Estimate the price of this product based on the description. Respond with the price in USD, no explanation\n\n "
               f"{item.summary}")
    return [
        {"role": "user", "content": message},
    ]


In [ ]:
# ========== 抽查：看 test[0] 的 messages 长什么样 ==========
test_messages_for(test[0])


In [ ]:
# ========== 用微调模型估一条价，再在全量 test 上 evaluate ==========

def gpt_4__1_nano_fine_tuned(item):
    # chat.completions：模型名用上格拿到的 fine_tuned_model_name
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        # 价格很短，限制 max_tokens 省钱也防跑题
        max_tokens=7
    )
    return response.choices[0].message.content


# 先肉眼对比：真价 vs 模型回复
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))
# 课程统一评估入口
evaluate(gpt_4__1_nano_fine_tuned, test)
